In [1]:
import cv2
import mediapipe as mp
import numpy as np

mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(
    static_image_mode=False, 
    model_complexity=1, # 1 is balanced for CPU-based real-time use
    min_detection_confidence=0.5, 
    min_tracking_confidence=0.5
)

I0000 00:00:1778376741.123224 27648478 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [2]:
# Define these globally so the model knows the input size
FACE_INDICES = [33, 133, 362, 263, 61, 291, 13, 14, 105, 334] 

# Static points: Pose(33) + Face(len(FACE_INDICES)) + L_Hand(21) + R_Hand(21)
num_static_points = 33 + len(FACE_INDICES) + 21 + 21
# (x,y,z) * 2 (for Velocity)
input_features = (num_static_points * 3) * 2

def extract_landmarks(results):
    # Pose: 33 points
    if results.pose_landmarks:
        pose_coords = np.array([[res.x, res.y, res.z] for res in results.pose_landmarks.landmark])
        mid_shoulder = (pose_coords[11] + pose_coords[12]) / 2
        shoulder_dist = np.linalg.norm(pose_coords[11] - pose_coords[12])
        pose = ((pose_coords - mid_shoulder) / (shoulder_dist + 1e-6)).flatten()
    else:
        pose = np.zeros(33*3)

    # Hands: 21 points each
    def process_hand(hand_results):
        if not hand_results: return np.zeros(21*3)
        coords = np.array([[res.x, res.y, res.z] for res in hand_results.landmark])
        wrist = coords[0]
        hand_size = np.linalg.norm(coords[0] - coords[9])
        return ((coords - wrist) / (hand_size + 1e-6)).flatten()

    lh = process_hand(results.left_hand_landmarks)
    rh = process_hand(results.right_hand_landmarks)

    # Face: selected indices
    if results.face_landmarks:
        face_all = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark])
        face = (face_all[FACE_INDICES] - face_all[1]).flatten()
    else:
        face = np.zeros(len(FACE_INDICES)*3)

    return np.concatenate([pose, face, lh, rh])

def extract_refined_features(results, prev_landmarks=None):
    # Get current landmarks
    current_landmarks = extract_landmarks(results) 
    
    # If landmarks are all zeros (no detection), velocity must be zero
    if prev_landmarks is None or not np.any(current_landmarks):
        # Initial frame: velocity is zero
        velocity = np.zeros_like(current_landmarks)
    else:
        # Calculate Delta (Velocity) only if tracking is valid
        velocity = current_landmarks - prev_landmarks
    
    # Concatenate spatial position + velocity
    return np.concatenate([current_landmarks, velocity])

def build_sequence(video_path, sequence_length=30):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Calculate skip rate to evenly sample 'sequence_length' frames
    skip_rate = max(1, total_frames // sequence_length)
    sequence = []
    prev_landmarks = None
    frame_count = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        # Follow paper's "Uniform Sampling" (Section 3.4) [cite: 174]
        if frame_count % skip_rate == 0:
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(image)
            
            # Extract features (position + velocity)
            current_raw = extract_landmarks(results)
            
            # Use current_raw to get velocity relative to prev_landmarks
            feat = extract_refined_features(results, prev_landmarks)
            sequence.append(feat)
            
            # Update prev_landmarks only if detection was successful
            # to maintain a coherent velocity delta for the next jump.
            if np.any(current_raw): 
                prev_landmarks = current_raw
        
        frame_count += 1
        if len(sequence) == sequence_length: break
            
    cap.release()
    
    # Paper-style padding: if video is short, pad with zeros [cite: 189]
    if len(sequence) < sequence_length:
        padding = [np.zeros(input_features)] * (sequence_length - len(sequence))
        sequence.extend(padding)
        
    return np.array(sequence)

In [3]:
import json
import os

# Load the target subset (nslt_300 is dict of {vid: {metadata}})
subset_file = 'WLASL_Dataset/nslt_300.json' 
with open(subset_file, 'r') as f:
    subset_data = json.load(f)
    # The keys are the video IDs we want
    target_ids = set(subset_data.keys()) 

# Load the main metadata to get the Gloss-to-ID mapping
with open('WLASL_Dataset/WLASL_v0.3.json', 'r') as f:
    wlasl_data = json.load(f)

# Load the list of IDs confirmed to be missing
missing_ids = set()
if os.path.exists('WLASL_Dataset/missing.txt'):
    with open('WLASL_Dataset/missing.txt', 'r') as f:
        missing_ids = set(f.read().splitlines())

# Build the filtered gloss map
gloss_map = {}
for entry in wlasl_data:
    gloss = entry['gloss']
    valid_instances = [
        inst['video_id'] for inst in entry['instances'] 
        if inst['video_id'] in target_ids and 
        inst['video_id'] not in missing_ids and 
        os.path.exists(f"WLASL_Dataset/videos/{inst['video_id']}.mp4")
    ]
    if valid_instances:
        gloss_map[gloss] = valid_instances

all_glosses = sorted(list(gloss_map.keys()))
gloss_to_idx = {gloss: i + 1 for i, gloss in enumerate(all_glosses)} # +1 for CTC blank
idx_to_gloss = {i: gloss for gloss, i in gloss_to_idx.items()}

print(f"Dataset ready! Total unique glosses: {len(all_glosses)}")
print(f"Total valid video samples: {sum(len(v) for v in gloss_map.values())}")

Dataset ready! Total unique glosses: 300
Total valid video samples: 2660


W0000 00:00:1778376741.176672 27648594 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778376741.185204 27648601 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778376741.186829 27648604 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778376741.186863 27648599 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778376741.186909 27648594 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778376741.189506 27648598 inference_feedback_manager.cc:114] Feedback ma

In [4]:
import torch
import torch.nn as nn

class ASLSignToGlossModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_glosses, dropout=0.2):
        super(ASLSignToGlossModel, self).__init__()
        
        # Temporal Convolution: Captures motion patterns across 5-frame windows
        # Input: (Batch, Features, Time)
        self.temporal_conv = nn.Sequential(
            nn.Conv1d(input_dim, hidden_dim, kernel_size=5, padding=2),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # BiLSTM: Models the full sequence forward and backward
        self.lstm = nn.LSTM(
            input_size=hidden_dim, 
            hidden_size=hidden_dim, 
            num_layers=2, 
            batch_first=True, 
            bidirectional=True, 
            dropout=dropout
        )
        
        # CTC Classifier: num_glosses + 1 for the 'blank' token
        self.classifier = nn.Linear(hidden_dim * 2, num_glosses + 1)
        self.softmax = nn.LogSoftmax(dim=2)

    def forward(self, x):
        # x shape: (Batch, Time, Features)
        # 1. Prep for Conv1d: (Batch, Features, Time)
        x = x.permute(0, 2, 1)
        x = self.temporal_conv(x)
        
        # 2. Prep for LSTM: (Batch, Time, Hidden)
        x = x.permute(0, 2, 1)
        lstm_out, _ = self.lstm(x)
        
        # 3. Output Logits
        logits = self.classifier(lstm_out)
        return self.softmax(logits)

In [5]:
def decode_batch_predictions(outputs, idx_to_gloss):
    """
    Performs greedy decoding on a batch of CTC outputs.
    Expects outputs shape: (Batch, Time, Classes)
    """
    # Get the most likely index at each time step for the whole batch
    arg_maxes = torch.argmax(outputs, dim=-1) # Shape: (Batch, Time)
    
    batch_decoded = []
    
    # Iterate through each video in the batch
    for i in range(arg_maxes.size(0)):
        decoded_indices = []
        previous_idx = -1
        
        for idx in arg_maxes[i]:
            idx = idx.item()
            # CTC Rule: Collapse repeats and ignore blank (index 0) 
            if idx != previous_idx and idx != 0:
                decoded_indices.append(idx)
            previous_idx = idx
            
        # Map indices to actual text glosses
        batch_decoded.append([idx_to_gloss[idx] for idx in decoded_indices])
        
    return batch_decoded

In [6]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    # Sort by length for CTC efficiency (optional but helpful)
    batch.sort(key=lambda x: len(x[0]), reverse=True)
    sequences, labels = zip(*batch)
    
    # Pad sequences to the same length (time_steps)
    sequences_padded = pad_sequence(sequences, batch_first=True)
    
    # Targets must be 1D for standard CTC or padded for batching
    labels = torch.stack(labels) 
    
    return sequences_padded, labels

In [7]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import os

# Hyperparameters from Paper (Section 3.6)
BATCH_SIZE = 64
HIDDEN_DIM = 512
EPOCHS = 80
initial_lr = 1e-4

# Update dataset to just load the file
class WLASLDataset(Dataset):
    def __init__(self, video_paths, labels):
        self.video_paths = video_paths
        self.labels = labels

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        vid_id = self.video_paths[idx]
        # Load precomputed features
        features = np.load(f"processed_features/{vid_id}.npy")
        return torch.tensor(features, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)

In [8]:
def precompute_features(gloss_map, save_dir="processed_features"):
    if not os.path.exists(save_dir): os.makedirs(save_dir)
    
    for gloss, video_ids in tqdm(gloss_map.items()):
        for vid in video_ids:
            video_path = f"WLASL_Dataset/videos/{vid}.mp4"
            if os.path.exists(video_path):
                try:
                    # Extract once
                    features = build_sequence(video_path) 
                    
                    # Only save if we actually got data back
                    if features is not None and len(features) > 0:
                        np.save(f"{save_dir}/{vid}.npy", features)
                    else:
                        print(f"\nSkipped {vid}: No frames extracted.")
                        
                except Exception as e:
                    print(f"\nCorrupted video {vid} - Error: {e}")

print("Precomputing MediaPipe features...")
# precompute_features(gloss_map)  # <--- ONLY RUN THIS ONCE IT TAKES SUPER LONGGGGG!!!

Precomputing MediaPipe features...


In [9]:
video_ids = []
labels = []

for gloss, vids in gloss_map.items():
    for vid in vids:
        # Only add if the precomputed feature file exists
        if os.path.exists(f"processed_features/{vid}.npy"):
            video_ids.append(vid)
            labels.append(gloss_to_idx[gloss]) # Assuming gloss_to_idx is defined earlier

train_ids, val_ids, train_labels, val_labels = train_test_split(
    video_ids, labels, test_size=0.15, random_state=42, stratify=labels
)

# Instantiate Datasets
train_dataset = WLASLDataset(train_ids, train_labels)
val_dataset = WLASLDataset(val_ids, val_labels)

# Define the Loaders
# Note: Ensure collate_fn is defined in a cell above this!
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)

print(f"Loaders initialized: {len(train_loader)} training batches.")

Loaders initialized: 36 training batches.


In [10]:
import os
import torch
import torch.nn as nn

# Device setup (M4 Pro GPU)
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
model = ASLSignToGlossModel(input_features, HIDDEN_DIM, len(all_glosses)).to(device)

# Force the loss function to live on the CPU
criterion = nn.CTCLoss(blank=0, zero_infinity=True).to("cpu") 
optimizer = torch.optim.Adam(model.parameters(), lr=initial_lr, weight_decay=1e-3)

# MultiStepLR handles the 40/60 epoch decay
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[40, 60], gamma=0.2)

def run_training():
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0
        for videos, targets in train_loader:
            # Send data to M4 GPU
            videos, targets = videos.to(device), targets.to(device)
            
            optimizer.zero_grad()
            
            # Neural network math happens blazing fast on MPS
            outputs = model(videos).permute(1, 0, 2) 

            batch_size = videos.size(0)
            seq_len = videos.size(1)

            # Keep lengths on CPU (CTCLoss requires CPU tensors for lengths anyway)
            input_lengths = torch.full((batch_size,), seq_len, dtype=torch.long)
            target_lengths = torch.full((batch_size,), 1, dtype=torch.long)

            # Explicitly cast outputs and targets to CPU just for the loss calculation
            loss = criterion(outputs.cpu(), targets.cpu(), input_lengths, target_lengths)
            
            if torch.isfinite(loss):
                # Gradients flow back to the MPS tensors automatically
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
        
        scheduler.step()
        print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {epoch_loss/len(train_loader):.4f}")

In [11]:
# Start the Sign-to-Gloss Training Stage
print(f"Starting training on device: {device}")
run_training()

Starting training on device: mps
Epoch 1/80 - Loss: 32.9975
Epoch 2/80 - Loss: 6.9488
Epoch 3/80 - Loss: 6.6465
Epoch 4/80 - Loss: 6.6053
Epoch 5/80 - Loss: 6.4960
Epoch 6/80 - Loss: 6.2238
Epoch 7/80 - Loss: 5.9353
Epoch 8/80 - Loss: 5.7939
Epoch 9/80 - Loss: 5.7370
Epoch 10/80 - Loss: 5.7139
Epoch 11/80 - Loss: 5.6986
Epoch 12/80 - Loss: 5.6794
Epoch 13/80 - Loss: 5.6613
Epoch 14/80 - Loss: 5.6372
Epoch 15/80 - Loss: 5.5926
Epoch 16/80 - Loss: 5.5140
Epoch 17/80 - Loss: 5.4126
Epoch 18/80 - Loss: 5.3095
Epoch 19/80 - Loss: 5.2093
Epoch 20/80 - Loss: 5.1111
Epoch 21/80 - Loss: 5.0197
Epoch 22/80 - Loss: 4.9324
Epoch 23/80 - Loss: 4.8423
Epoch 24/80 - Loss: 4.7420
Epoch 25/80 - Loss: 4.6414
Epoch 26/80 - Loss: 4.5531
Epoch 27/80 - Loss: 4.4283
Epoch 28/80 - Loss: 4.2975
Epoch 29/80 - Loss: 4.1517
Epoch 30/80 - Loss: 4.0524
Epoch 31/80 - Loss: 3.8915
Epoch 32/80 - Loss: 3.7584
Epoch 33/80 - Loss: 3.6104
Epoch 34/80 - Loss: 3.4719
Epoch 35/80 - Loss: 3.3485
Epoch 36/80 - Loss: 3.1956
Epo